# Project 1 ANSC 4040 Data Split

This is a notebook for Project 1 that will be used for making the test, train and validation data split. The plan outlined below organizes the split chronologically by date and by cow. With the use of the tree-based model I plan to use, it it vital that all test cows are prepresented in the training at least once in order to improve the accuract of the model. 

In [1]:
#import pandas as pd
import pandas as pd

In [2]:
# Add file path and load the dataset
path = 'clean_data.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-8.839529e+18,1.0,365.0,Pregnant,2019-12-09,2.902991,0.898113,4.975908,11.158372,226,1
1,-5.365332e+18,1.0,66.0,Bred,2021-07-12,3.311224,1.401600,5.347854,15.059267,268,2
2,NaN,NaN,NaN,NaN,2020-08-24,3.220506,3.501733,7.547777,14.560315,270,1
3,7.750424e+18,1.0,244.0,Pregnant,2019-12-04,3.900894,4.100475,8.400531,15.331422,231,3
4,NaN,NaN,NaN,NaN,2021-03-21,4.218409,5.098378,9.198853,13.970645,198,2


# Plan for the split

1. Exlude missing values into a new data set- "missing_data"
2. Sort by date and by cow
3. Split by time order, ensuring each cow present in the test set is represented in the training set
     Keep the most recent dates in the test set (10%)
     Use earlier dates for training (70%)
     Use a validation window between them (20%)

# Exlude missing values into a new data set - "missing_data"

In [7]:
# Remove rows with any missing values from the clean dataset
before_rows = len(data)

data = data.dropna().copy()

after_rows = len(data)

print(f"Rows before removing missing values: {before_rows}")
print(f"Rows after removing missing values: {after_rows}")
print(f"Rows removed: {before_rows - after_rows}")

# Keep rows with missing values in a separate dataset
missing_data = data[data.isna().any(axis=1)].copy()

# Keep only rows without missing values
clean_data = data.dropna().copy()

missing_data.to_csv("missing_data.csv", index=False)
clean_data.to_csv("clean_data_no_missing.csv", index=False)


Rows before removing missing values: 8285371
Rows after removing missing values: 6599539
Rows removed: 1685832


# Sort Remaining Data by Date

In [8]:
# Add file path and load the dataset without any mising values
path = 'clean_data_no_missing.csv'
data = pd.read_csv(path)
data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-5.671649e+18,2.0,439.0,Open,2019-06-14,2.812273,3.098036,6.323078,6.985322,147,3
1,-8.610662e+18,4.0,36.0,Fresh,2019-06-14,3.719457,3.900894,7.416235,19.232316,308,1
2,-8.077827e+18,1.0,37.0,Fresh,2019-06-14,3.311224,2.299713,6.023707,13.335616,240,2
3,3.786853e+18,1.0,309.0,Pregnant,2019-06-14,4.309128,5.297959,10.523343,14.288160,199,3
4,8.643179e+18,2.0,53.0,Open,2019-06-14,4.717361,5.297959,9.770380,17.735462,226,1


In [9]:
# Sort the dataset by the date column in chronological order
date_columns = [col for col in data.columns if "date" in col.lower()]

if not date_columns:
    raise ValueError("No date column found. Please check the dataset column names.")

date_col = date_columns[0]
data[date_col] = pd.to_datetime(data[date_col], errors="coerce")

# Sort by date and reset index
data = data.sort_values(by=date_col, ascending=True).reset_index(drop=True)

data.head()

,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-5.671649e+18,2.0,439.0,Open,2019-06-14,2.812273,3.098036,6.323078,6.985322,147,3
1,-2.837933e+18,2.0,120.0,Bred,2019-06-14,4.309128,5.502075,10.550559,17.735462,243,2
2,-4.652668e+18,1.0,33.0,Fresh,2019-06-14,3.900894,4.799007,9.126278,12.745946,193,1
3,4.607693e+18,1.0,189.0,Bred,2019-06-14,3.810176,4.000685,9.198853,12.246994,192,1
4,4.565455e+17,1.0,42.0,Fresh,2019-06-14,4.082331,4.300056,7.974154,16.238607,237,1


# Split by time order only
Keep the most recent dates in the test set (10%)
Use earlier dates for training (70%)
Use a validation window between them (20%) 

In [11]:
# Compute split sizes
n = len(data)
test_size = int(round(n * 0.10))
val_size = int(round(n * 0.20))
train_size = n - test_size - val_size

# Ensure valid sizes
if train_size <= 0:
    raise ValueError("Not enough rows to create all three splits.")

# Create split boundaries
train_end = train_size
val_end = train_size + val_size

train = data.iloc[:train_end].copy()
val = data.iloc[train_end:val_end].copy()
test = data.iloc[val_end:].copy()

print(f"Total rows: {n}")
print(f"Train rows: {len(train)}")
print(f"Validation rows: {len(val)}")
print(f"Test rows: {len(test)}")

train.head()

Total rows: 6599539
Train rows: 4619677
Validation rows: 1319908
Test rows: 659954


,AnimalId,LactationNumber,DaysInMilk,ReproductionStatus,EventDate,Avgmilkflow,Flow30_60Session,YieldFirst2Min_Session,YieldSession,DurationSession_sec,milking
0,-5.671649e+18,2.0,439.0,Open,2019-06-14,2.812273,3.098036,6.323078,6.985322,147,3
1,-2.837933e+18,2.0,120.0,Bred,2019-06-14,4.309128,5.502075,10.550559,17.735462,243,2
2,-4.652668e+18,1.0,33.0,Fresh,2019-06-14,3.900894,4.799007,9.126278,12.745946,193,1
3,4.607693e+18,1.0,189.0,Bred,2019-06-14,3.810176,4.000685,9.198853,12.246994,192,1
4,4.565455e+17,1.0,42.0,Fresh,2019-06-14,4.082331,4.300056,7.974154,16.238607,237,1


In [12]:
# Save the train, validation, and test splits as separate CSV files
train.to_csv("train_data.csv", index=False)
val.to_csv("val_data.csv", index=False)
test.to_csv("test_data.csv", index=False)

print("Saved: train_data.csv, val_data.csv, test_data.csv")

Saved: train_data.csv, val_data.csv, test_data.csv
